**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

In [1]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc

def compare_quantization():
    print("🔬 정확한 모델 크기 비교\n")
    model_name = "gpt2"

    # 1. 원본 모델 (FP32)
    print("=" * 50)
    print("1. 원본 모델 (FP32)")
    print("=" * 50)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model_fp32 = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")
    fp32_params = sum(p.numel() for p in model_fp32.parameters())
    fp32_size = sum(p.numel() * p.element_size() for p in model_fp32.parameters()) / (1024**2)
    gpu_fp32 = torch.cuda.memory_allocated() / (1024**2)

    print(f"📊 파라미터 수: {fp32_params:,}")
    print(f"📊 이론적 크기: {fp32_size:.2f} MB")
    print(f"🎮 GPU 메모리: {gpu_fp32:.2f} MB")

    del model_fp32
    gc.collect()
    torch.cuda.empty_cache()

    # 2. FP16 모델
    print("\n" + "=" * 50)
    print("2. FP16 모델")
    print("=" * 50)

    model_fp16 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda")
    gpu_fp16 = torch.cuda.memory_allocated() / (1024**2)

    print(f"📊 예상 크기: {fp32_size/2:.2f} MB")
    print(f"🎮 GPU 메모리: {gpu_fp16:.2f} MB")
    print(f"📉 감소율: {(1 - gpu_fp16/gpu_fp32)*100:.1f}%")

    del model_fp16
    gc.collect()
    torch.cuda.empty_cache()

    # 3. 8-bit 양자화
    print("\n" + "=" * 50)
    print("3. 8-bit 양자화 모델")
    print("=" * 50)

    bnb_config_8bit = BitsAndBytesConfig(load_in_8bit=True)
    model_8bit = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config_8bit,
        device_map={"": 0}
    )
    gpu_8bit = torch.cuda.memory_allocated() / (1024**2)

    print(f"📊 예상 크기: {fp32_size/4:.2f} MB")
    print(f"🎮 GPU 메모리: {gpu_8bit:.2f} MB")
    print(f"📉 감소율: {(1 - gpu_8bit/gpu_fp32)*100:.1f}%")

    del model_8bit
    gc.collect()
    torch.cuda.empty_cache()

    # 4. 4-bit 양자화
    print("\n" + "=" * 50)
    print("4. 4-bit 양자화 모델 (NF4)")
    print("=" * 50)

    bnb_config_4bit = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    model_4bit = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config_4bit,
        device_map={"": 0}
    )
    gpu_4bit = torch.cuda.memory_allocated() / (1024**2)

    print(f"📊 예상 크기: {fp32_size/8:.2f} MB")
    print(f"🎮 GPU 메모리: {gpu_4bit:.2f} MB")
    print(f"📉 감소율: {(1 - gpu_4bit/gpu_fp32)*100:.1f}%")

    del model_4bit
    gc.collect()
    torch.cuda.empty_cache()

    # 요약
    print("\n" + "=" * 50)
    print("📊 요약 (GPU 실측)")
    print("=" * 50)
    print(f"{'모델':<10} {'GPU 메모리':>12} {'감소율':>10}")
    print("-" * 35)
    print(f"{'FP32':<10} {gpu_fp32:>10.2f} MB {'기준':>10}")
    print(f"{'FP16':<10} {gpu_fp16:>10.2f} MB {(1-gpu_fp16/gpu_fp32)*100:>8.1f}%")
    print(f"{'INT8':<10} {gpu_8bit:>10.2f} MB {(1-gpu_8bit/gpu_fp32)*100:>8.1f}%")
    print(f"{'INT4':<10} {gpu_4bit:>10.2f} MB {(1-gpu_4bit/gpu_fp32)*100:>8.1f}%")

compare_quantization()

/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔬 정확한 모델 크기 비교

1. 원본 모델 (FP32)
📊 파라미터 수: 124,439,808
📊 이론적 크기: 474.70 MB
🎮 GPU 메모리: 487.47 MB

2. FP16 모델


`torch_dtype` is deprecated! Use `dtype` instead!


📊 예상 크기: 237.35 MB
🎮 GPU 메모리: 255.49 MB
📉 감소율: 47.6%

3. 8-bit 양자화 모델
📊 예상 크기: 118.68 MB
🎮 GPU 메모리: 169.05 MB
📉 감소율: 65.3%

4. 4-bit 양자화 모델 (NF4)
📊 예상 크기: 59.34 MB
🎮 GPU 메모리: 132.94 MB
📉 감소율: 72.7%

📊 요약 (GPU 실측)
모델              GPU 메모리        감소율
-----------------------------------
FP32           487.47 MB         기준
FP16           255.49 MB     47.6%
INT8           169.05 MB     65.3%
INT4           132.94 MB     72.7%
